# Parking-space occupancy classification

This notebook trains a compact CNN to classify a pre-cropped parking-space image as `free` or `busy`. CNRPark-Patches has one space per image and no bounding boxes, so this is classification rather than object detection. Consequently, mAP@0.50 is not applicable.

Run from top to bottom. On a fresh Colab runtime, the first cell clones this public repository; locally, it uses the current project folder.

In [ ]:
from pathlib import Path
import os, subprocess, urllib.request, zipfile

# Clone the repository only when the notebook is running outside its checkout.
if not Path('src/train.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/Alihansagat/Deep-Learning.git'], check=True)
    os.chdir(Path('Deep-Learning') / 'Project_Parking')

# Install pinned dependencies for a reproducible fresh runtime.
%pip install -r requirements.txt

print('Project root:', Path.cwd())

## Download and inspect the dataset

The small CNRPark patch archive is about 36.6 MB. It is distributed from the release linked by the dataset's official page. The archive is extracted locally in this runtime and is not committed to GitHub.

In [ ]:
DATA_DIR = Path('CNRPark-Patches-150x150')
if not (DATA_DIR / 'A').exists():
    archive_path = Path('CNRPark-Patches-150x150.zip')
    if not archive_path.exists():
        url = 'https://github.com/fabiocarrara/deep-parking/releases/download/archive/CNRPark-Patches-150x150.zip'
        print('Downloading the small CNRPark archive...')
        urllib.request.urlretrieve(url, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall('.')
    # Some ZIP tools may nest the dataset under an extra directory. Find it.
    candidates = list(Path('.').glob('**/CNRPark-Patches-150x150/A'))
    if candidates:
        DATA_DIR = candidates[0].parent
    elif Path('A/free').exists() and Path('B/busy').exists():
        DATA_DIR = Path('.')

from collections import Counter
images = sorted(DATA_DIR.glob('*/*/*.jpg'))
print('Dataset path:', DATA_DIR.resolve())
print('Total images:', len(images))
print('Classes:', Counter(p.parent.name for p in images))
print('Cameras:', Counter(p.parent.parent.name for p in images))
assert len(images) > 0 and {p.parent.name for p in images} == {'free', 'busy'}

## Ground-truth examples

Each crop is already a single parking space; its folder supplies the ground-truth class. These examples show the labeled crops (there are no bounding boxes in this dataset).

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
random.seed(1409)
examples = random.sample(images, min(10, len(images)))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, path in zip(axes.flat, examples):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(f'{path.parent.name} | camera {path.parent.parent.name}')
    ax.axis('off')
plt.tight_layout(); plt.show()

## Train baseline and controlled experiment

Hypothesis: Adam with learning rate `0.0001` may improve validation macro-F1 over `0.001`. Only learning rate changes. The split, seed, model, image size, batch size, augmentation, and epoch count remain fixed. The earlier date is used for train/validation and the later date for the held-out test; with only two date-camera pairs, this test combines date and camera shift.

In [ ]:
# Run both experiments; selected checkpoint is chosen using validation only.
!python src/train.py --data-dir "{DATA_DIR}" --output-dir outputs --seed 1409 --epochs 12 --batch-size 64 --image-size 150 --baseline-lr 0.001 --experiment-lr 0.0001

## Compare runs and inspect final test results

The code evaluates the selected checkpoint on the held-out test set only after choosing the run by validation macro-F1. `average_precision_free` is classification average precision, not detection mAP@0.50.

In [ ]:
import json
from IPython.display import display, Image as DisplayImage
from IPython.display import Markdown
results = json.loads(Path('outputs/results.json').read_text())
rows = []
for run, data in results['runs'].items():
    m = data['validation_metrics']
    rows.append({'run': run, 'learning_rate': data['learning_rate'], 'val_precision_free': m['precision_free'], 'val_recall_free': m['recall_free'], 'val_macro_f1': m['macro_f1'], 'train_seconds': data['training_seconds']})
for row in rows:
    print(row)
print('Selected run:', results['selected_run_by_validation_macro_f1'])
print('Held-out test metrics:', json.dumps(results['held_out_test_metrics'], indent=2))
print('Detection metric:', results['detection_metric_note'])
display(DisplayImage(filename='outputs/test_examples.png'))